# 08 · Paged KV allocator: block tables, reference counts, copy-on-write

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path steps 7–8. Sources: Kwon et al., SOSP 2023; github.com/vllm-project/vllm.

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Drive `inference_lab.paged` by hand, watch block tables form, translate token positions to
physical slots, and trigger copy-on-write.

## Theory and equations
$\text{slot}(r,p) = \text{table}_r[\lfloor p/B\rfloor]\cdot B + (p \bmod B)$, and
$\text{ref}(b) = |\{(r,i): \text{table}_r[i] = b\}|$. A block returns to the free list when its refcount reaches 0.

## Diagram
```
seq "a": logical [0][1][2]  -> table [0, 2, 5]      physical pool: 0:a 1:b 2:a 3:b 4:- 5:a ...
```

## Implementation

In [ ]:
from inference_lab.paged import PagedKVCache, OutOfBlocks
c = PagedKVCache(num_blocks=12, block_size=4)
c.add("a", 5); c.add("b", 7)
for _ in range(4): c.append_token("a")
print("tables:", {k: v.table for k, v in c.seqs.items()}, "free:", c.alloc.num_free)
print("token 8 of a lives at (block, offset) =", c.physical_slot("a", 8))
c.check_invariants()

## Experiment: fork and copy-on-write (*measured*)

In [ ]:
c.fork("b", "b2")                           # parallel sample shares all of b's blocks
print("refcounts after fork:", c.alloc.refcount)
c.append_token("b2")                         # writes into the shared, partially filled last block
print("copy-on-write events:", c.alloc.copies, " b:", c.seqs["b"].table, " b2:", c.seqs["b2"].table)
c.free("b"); print("after freeing b, refcounts:", c.alloc.refcount)
c.check_invariants()

## Results: running out of blocks

In [ ]:
try:
    while True:
        c.append_token("a")
except OutOfBlocks as e:
    print("OutOfBlocks:", e, "| a holds", c.seqs["a"].tokens, "tokens in", len(c.seqs["a"].table), "blocks")

## Interpretation
Physical placement is arbitrary; the table preserves order. Sharing is a refcount increment,
and the first divergent write pays one block copy. Exhaustion surfaces as an exception the
scheduler must handle (preempt, swap, or queue).

## Limitations
No GPU memory; the pool is a Python list. Real engines batch these operations per iteration.

## Conclusion
A paged KV manager is a small amount of bookkeeping with strong invariants. The tests check
those invariants after hundreds of random operations.